# Module 00: Environment Setup and Data Access

## Learning Objectives
By completing this notebook, you will:
1. Verify all required packages are installed and working
2. Configure FRED API access (or confirm CSV fallback)
3. Download and inspect the core datasets used throughout the course
4. Understand the data conventions used in all subsequent notebooks

## Prerequisites
- Python 3.9+
- Basic pandas familiarity

## Estimated Time: 15 minutes

---

**Note on FRED API key:** A free key is available at https://fred.stlouisfed.org/docs/api/api_key.html  
All notebooks work without a key using CSV fallback files in `resources/`.

## 1. Package Verification

Run the cell below to verify all required packages. If any fail, install with `pip install <package>`.

In [ ]:
# Core scientific stack
import importlib
import sys

REQUIRED_PACKAGES = [
    ('numpy', 'numpy'),
    ('pandas', 'pandas'),
    ('scipy', 'scipy'),
    ('matplotlib', 'matplotlib'),
    ('statsmodels', 'statsmodels'),
    ('sklearn', 'scikit-learn'),
    ('yfinance', 'yfinance'),
]

OPTIONAL_PACKAGES = [
    ('fredapi', 'fredapi'),
    ('plotly', 'plotly'),
    ('ipywidgets', 'ipywidgets'),
]

print("Checking required packages...")
all_ok = True
for import_name, pip_name in REQUIRED_PACKAGES:
    try:
        mod = importlib.import_module(import_name)
        version = getattr(mod, '__version__', 'unknown')
        print(f"  OK  {pip_name:<20} version {version}")
    except ImportError:
        print(f"  MISSING  {pip_name} -- run: pip install {pip_name}")
        all_ok = False

print("\nChecking optional packages...")
for import_name, pip_name in OPTIONAL_PACKAGES:
    try:
        mod = importlib.import_module(import_name)
        version = getattr(mod, '__version__', 'unknown')
        print(f"  OK  {pip_name:<20} version {version}")
    except ImportError:
        print(f"  OPTIONAL (not installed): {pip_name}")

print(f"\nPython version: {sys.version}")
if all_ok:
    print("\nAll required packages are available.")
else:
    print("\nSome required packages are missing. Install them before continuing.")

## 2. FRED API Configuration

Set your FRED API key below. If you don't have one, set `USE_FRED = False` and the notebook will use CSV fallback files.

To get a free FRED API key:
1. Go to https://fred.stlouisfed.org/
2. Create a free account
3. Navigate to My Account → API Keys → Request API Key
4. Paste the key in the cell below

In [ ]:
import os
import pandas as pd
import numpy as np

# Configuration: set USE_FRED = False to use CSV fallbacks without an API key
USE_FRED = False  # Change to True and set FRED_API_KEY if you have a key

# Option 1: Set directly here (for local use only — don't commit API keys)
FRED_API_KEY = None  # Replace with your key: 'abcd1234...'

# Option 2: Read from environment variable (recommended for shared environments)
if FRED_API_KEY is None:
    FRED_API_KEY = os.environ.get('FRED_API_KEY')

if USE_FRED and FRED_API_KEY is not None:
    try:
        from fredapi import Fred
        fred = Fred(api_key=FRED_API_KEY)
        # Quick test: fetch one observation
        test = fred.get_series('GDPC1', observation_start='2023-01-01', observation_end='2023-12-31')
        print(f"FRED API connection successful. Latest GDP observation: {test.index[-1].date()}")
    except Exception as e:
        print(f"FRED API connection failed: {e}")
        print("Switching to CSV fallback.")
        USE_FRED = False
elif USE_FRED and FRED_API_KEY is None:
    print("USE_FRED=True but no API key provided. Switching to CSV fallback.")
    USE_FRED = False
else:
    print("Using CSV fallback files (no FRED API key required).")

print(f"\nConfiguration: USE_FRED = {USE_FRED}")

## 3. Load Core Datasets

We load three datasets used throughout the course:
1. **Quarterly GDP growth** — our primary target variable
2. **Monthly industrial production growth** — our primary monthly regressor  
3. **Daily S&P 500 returns** — our primary daily regressor

The `load_series()` function tries FRED first, then falls back to CSV.

In [ ]:
import os

# Determine the resources directory (relative to this notebook)
NOTEBOOK_DIR = os.path.dirname(os.path.abspath('__file__')) if '__file__' in dir() else os.getcwd()
RESOURCES_DIR = os.path.join(os.path.dirname(NOTEBOOK_DIR), 'resources')

# If running from the notebooks/ subdirectory, adjust path
if not os.path.exists(RESOURCES_DIR):
    RESOURCES_DIR = '../resources'
if not os.path.exists(RESOURCES_DIR):
    RESOURCES_DIR = 'resources'

def load_series(series_id, csv_filename, start_date='2000-01-01'):
    """
    Load a data series from FRED or local CSV fallback.
    
    Parameters
    ----------
    series_id : str
        FRED series identifier (e.g., 'GDPC1').
    csv_filename : str
        Name of CSV file in resources directory.
    start_date : str
        Start date for FRED download.
    
    Returns
    -------
    pd.Series with DatetimeIndex
    """
    if USE_FRED and FRED_API_KEY:
        try:
            series = fred.get_series(series_id, observation_start=start_date)
            series.name = series_id
            return series
        except Exception as e:
            print(f"  FRED download failed for {series_id}: {e}")
            print(f"  Falling back to CSV: {csv_filename}")
    
    # CSV fallback
    csv_path = os.path.join(RESOURCES_DIR, csv_filename)
    series = pd.read_csv(csv_path, index_col='date', parse_dates=True).squeeze()
    series.name = series_id
    return series


# Load quarterly GDP
print("Loading quarterly GDP growth...")
gdp_raw = load_series('GDPC1', 'gdp_quarterly.csv')

# If loading the raw GDPC1 level, convert to growth rate
if gdp_raw.mean() > 100:  # It's a level series, not growth rate
    gdp_growth = gdp_raw.pct_change() * 100
    gdp_growth = gdp_growth.dropna()
else:  # Already growth rate from CSV
    gdp_growth = gdp_raw.dropna()

print(f"  Observations: {len(gdp_growth)}")
print(f"  Date range:   {gdp_growth.index[0].date()} to {gdp_growth.index[-1].date()}")
print(f"  Mean: {gdp_growth.mean():.3f}%  Std: {gdp_growth.std():.3f}%")

# Load monthly industrial production
print("\nLoading monthly industrial production growth...")
ip_raw = load_series('INDPRO', 'industrial_production_monthly.csv')

if ip_raw.mean() > 50:  # It's an index level
    ip_growth = ip_raw.pct_change() * 100
    ip_growth = ip_growth.dropna()
else:  # Already growth rate from CSV
    ip_growth = ip_raw.dropna()

print(f"  Observations: {len(ip_growth)}")
print(f"  Date range:   {ip_growth.index[0].date()} to {ip_growth.index[-1].date()}")
print(f"  Mean: {ip_growth.mean():.3f}%  Std: {ip_growth.std():.3f}%")

# Load daily S&P 500
print("\nLoading daily S&P 500 returns...")
if USE_FRED or True:  # Always try yfinance for equity data
    try:
        import yfinance as yf
        sp500_raw = yf.download('^GSPC', start='2000-01-01', progress=False)
        sp500_returns = sp500_raw['Adj Close'].pct_change().dropna() * 100
        sp500_returns.name = 'SP500'
        print(f"  Loaded via yfinance")
    except Exception as e:
        print(f"  yfinance failed: {e}, loading CSV fallback")
        sp500_returns = load_series('^GSPC', 'sp500_daily.csv')
else:
    sp500_returns = load_series('^GSPC', 'sp500_daily.csv')

print(f"  Observations: {len(sp500_returns)}")
print(f"  Date range:   {sp500_returns.index[0].date()} to {sp500_returns.index[-1].date()}")
print(f"  Mean: {sp500_returns.mean():.4f}%  Std: {sp500_returns.std():.3f}%")

## 4. Data Quality Checks

Before using any dataset in modeling, always run basic quality checks.

Key things to verify:
- No unexpected missing values
- Date range covers the period of interest
- Values are in expected range (growth rates, not index levels)
- COVID outliers are present (they're real data)

In [ ]:
def data_quality_report(series, name, expected_freq=None):
    """
    Print a comprehensive data quality summary.
    
    Parameters
    ----------
    series : pd.Series
        Series to report on.
    name : str
        Human-readable name for display.
    expected_freq : str or None
        Expected frequency string (e.g., 'Q', 'M', 'B') for verification.
    """
    print(f"=== {name} ===")
    print(f"  Observations:  {len(series)}")
    print(f"  Date range:    {series.index[0].date()} to {series.index[-1].date()}")
    print(f"  Missing:       {series.isna().sum()} ({series.isna().mean():.1%})")
    print(f"  Mean:          {series.mean():.4f}")
    print(f"  Std Dev:       {series.std():.4f}")
    print(f"  Min:           {series.min():.4f} ({series.idxmin().date()})")
    print(f"  Max:           {series.max():.4f} ({series.idxmax().date()})")
    
    # Identify large outliers (beyond 4 standard deviations)
    z_scores = (series - series.mean()) / series.std()
    outliers = series[z_scores.abs() > 4]
    if len(outliers) > 0:
        print(f"  Outliers (>4σ): {len(outliers)} observations")
        for date, val in outliers.items():
            print(f"    {date.date()}: {val:.4f}")
    print()

data_quality_report(gdp_growth, "Real GDP Growth (QoQ %)")
data_quality_report(ip_growth, "Industrial Production Growth (MoM %)")
data_quality_report(sp500_returns, "S&P 500 Daily Returns (%)")

## 5. Visualizing the Mixed-Frequency Structure

The plot below shows all three series on a common time axis, making the frequency mismatch visible. Quarterly GDP updates once per panel; monthly IP three times; daily S&P 500 ~65 times.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Focus on 2018–2024 to see the frequency structure clearly
start_plot = '2018-01-01'
end_plot = '2024-12-31'

gdp_plot = gdp_growth.loc[start_plot:end_plot]
ip_plot = ip_growth.loc[start_plot:end_plot]
sp500_plot = sp500_returns.loc[start_plot:end_plot]

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=False)
fig.suptitle('Mixed-Frequency Data: Three Layers of Economic Information',
             fontsize=14, fontweight='bold', y=1.02)

# Panel 1: Quarterly GDP
ax1 = axes[0]
ax1.bar(gdp_plot.index, gdp_plot.values, width=80,
        color=['steelblue' if v >= 0 else 'coral' for v in gdp_plot.values],
        alpha=0.8)
ax1.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax1.set_ylabel('QoQ Growth (%)')
ax1.set_title(f'Quarterly GDP Growth  |  {len(gdp_plot)} observations', fontsize=11)
ax1.set_xlim(pd.Timestamp(start_plot), pd.Timestamp(end_plot))
ax1.xaxis.set_major_locator(mdates.YearLocator())
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax1.text(0.02, 0.95, 'Frequency: Quarterly',
         transform=ax1.transAxes, fontsize=9, color='darkblue',
         verticalalignment='top')

# Panel 2: Monthly IP
ax2 = axes[1]
ax2.bar(ip_plot.index, ip_plot.values, width=25,
        color=['steelblue' if v >= 0 else 'coral' for v in ip_plot.values],
        alpha=0.8)
ax2.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax2.set_ylabel('MoM Growth (%)')
ax2.set_title(f'Monthly Industrial Production Growth  |  {len(ip_plot)} observations', fontsize=11)
ax2.set_xlim(pd.Timestamp(start_plot), pd.Timestamp(end_plot))
ax2.xaxis.set_major_locator(mdates.YearLocator())
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax2.text(0.02, 0.95, 'Frequency: Monthly (3× quarterly)',
         transform=ax2.transAxes, fontsize=9, color='darkblue',
         verticalalignment='top')

# Panel 3: Daily S&P 500
ax3 = axes[2]
ax3.fill_between(sp500_plot.index, sp500_plot.values, 0,
                  where=sp500_plot.values >= 0, color='steelblue', alpha=0.6)
ax3.fill_between(sp500_plot.index, sp500_plot.values, 0,
                  where=sp500_plot.values < 0, color='coral', alpha=0.6)
ax3.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax3.set_ylabel('Daily Return (%)')
ax3.set_title(f'Daily S&P 500 Returns  |  {len(sp500_plot)} observations', fontsize=11)
ax3.set_xlim(pd.Timestamp(start_plot), pd.Timestamp(end_plot))
ax3.xaxis.set_major_locator(mdates.YearLocator())
ax3.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax3.text(0.02, 0.95, 'Frequency: Daily (~65× quarterly)',
         transform=ax3.transAxes, fontsize=9, color='darkblue',
         verticalalignment='top')

plt.tight_layout()
plt.savefig('mixed_frequency_overview.png', dpi=120, bbox_inches='tight')
plt.show()
print("Figure saved to mixed_frequency_overview.png")

## 6. Frequency Alignment Demo

Before modeling, we need to map each high-frequency observation to its corresponding low-frequency period. The code below demonstrates the Period index approach used throughout the course.

Each quarterly GDP observation aligns with exactly 3 monthly IP observations.

In [ ]:
# Convert to Period index for unambiguous alignment
gdp_p = gdp_growth.copy()
gdp_p.index = pd.PeriodIndex(gdp_growth.index, freq='Q')

ip_p = ip_growth.copy()
ip_p.index = pd.PeriodIndex(ip_growth.index, freq='M')

# Show alignment for a recent quarter
example_quarter = pd.Period('2023Q3', freq='Q')
print(f"Example: Quarter {example_quarter}")
print(f"  GDP growth: {gdp_p.get(example_quarter, 'N/A'):.4f}%")
print()

# The three monthly observations that map to this quarter
months_in_quarter = pd.period_range(start=example_quarter.start_time,
                                     end=example_quarter.end_time,
                                     freq='M')
print("  Corresponding monthly IP observations:")
for month in months_in_quarter:
    val = ip_p.get(month, float('nan'))
    label = ['(Month 1 of quarter)', '(Month 2 of quarter)', '(Month 3 of quarter)'][list(months_in_quarter).index(month)]
    print(f"    {month}: IP growth = {val:.4f}%  {label}")

# Verify we have 3 monthly obs per quarterly obs in our sample
ip_by_quarter = ip_p.groupby(ip_p.index.to_timestamp().to_period('Q')).count()
print(f"\nMonthly observations per quarter:")
print(f"  All quarters have exactly 3? {(ip_by_quarter == 3).all()}")
print(f"  Counts: {ip_by_quarter.value_counts().to_dict()}")

## 7. Self-Check: Environment Verification

The checks below confirm your environment is ready for the course. All should pass before proceeding to Module 00, Notebook 02.

In [ ]:
print("Running environment verification...\n")

passed = 0
total = 0

def check(condition, description, hint=""):
    global passed, total
    total += 1
    if condition:
        print(f"  PASS  {description}")
        passed += 1
    else:
        print(f"  FAIL  {description}")
        if hint:
            print(f"        Hint: {hint}")

# Data availability checks
check(len(gdp_growth) > 0,
      "GDP growth series loaded",
      "Check resources/gdp_quarterly.csv exists")

check(len(ip_growth) > 0,
      "Industrial production series loaded",
      "Check resources/industrial_production_monthly.csv exists")

check(len(sp500_returns) > 0,
      "S&P 500 returns series loaded",
      "Check resources/sp500_daily.csv or yfinance connectivity")

# Data quality checks
check(gdp_growth.isna().sum() == 0,
      "GDP growth has no missing values",
      "Run gdp_growth.dropna() in the load step")

check(ip_growth.isna().sum() == 0,
      "IP growth has no missing values",
      "Run ip_growth.dropna() in the load step")

check(gdp_growth.std() < 10,
      "GDP growth values are in percentage form (not fraction)",
      "Check: gdp_growth should be ~-2% to +3%, not -0.02 to +0.03")

check(ip_growth.std() < 10,
      "IP growth values are in percentage form",
      "Check: ip_growth values should be ~-5% to +5%")

# Frequency checks
check(len(gdp_growth) < len(ip_growth),
      "GDP (quarterly) has fewer observations than IP (monthly)",
      "Something went wrong with frequency — check index types")

check(len(ip_growth) < len(sp500_returns),
      "IP (monthly) has fewer observations than S&P 500 (daily)",
      "Something went wrong with frequency — check index types")

# Package checks
try:
    from scipy.optimize import minimize
    check(True, "scipy.optimize.minimize available (needed for MIDAS NLS)")
except ImportError:
    check(False, "scipy.optimize.minimize available",
          "pip install scipy")

try:
    import statsmodels.api as sm
    check(True, "statsmodels available (needed for state space models)")
except ImportError:
    check(False, "statsmodels available",
          "pip install statsmodels")

print(f"\n{'='*40}")
print(f"Results: {passed}/{total} checks passed")
if passed == total:
    print("Environment is ready. Proceed to Notebook 02.")
else:
    print(f"{total - passed} check(s) failed. Resolve issues before continuing.")

## Summary

### What You've Set Up
1. **All required packages** installed and verified
2. **FRED API** configured (or CSV fallback confirmed)
3. **Three core datasets** loaded and quality-checked:
   - Quarterly GDP growth (2000–2024, ~100 observations)
   - Monthly IP growth (2000–2024, ~300 observations)
   - Daily S&P 500 returns (2000–2024, ~6,300 observations)
4. **Period indexing** demonstrated for frequency alignment

### Key Data Facts
- GDP and IP are expressed as **percentage growth rates**
- The COVID period (2020 Q1–Q2) contains extreme outliers — these are real data
- Monthly-to-quarterly ratio: **3:1**; daily-to-quarterly ratio: **~65:1**

### What's Next
Notebook 02 explores the information structure of these series — showing concretely how aggregation discards within-quarter timing information, motivating the MIDAS approach.

---

**Configuration used in this session:**

In [ ]:
print(f"USE_FRED:          {USE_FRED}")
print(f"FRED API key set:  {FRED_API_KEY is not None}")
print(f"Resources dir:     {RESOURCES_DIR}")
print(f"GDP observations:  {len(gdp_growth)}")
print(f"IP observations:   {len(ip_growth)}")
print(f"S&P500 obs:        {len(sp500_returns)}")